# Neural Network Training for MT5 Predictor

Interactive notebook for training and experimenting with neural networks for forex prediction.

**Advantages over script:**
- Interactive experimentation
- Visualize data and results inline
- Easy hyperparameter tuning
- Save/load checkpoints
- GPU acceleration with TensorFlow

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))} device(s)")
if tf.config.list_physical_devices('GPU'):
    print("✅ GPU acceleration enabled!")
else:
    print("⚠️ Running on CPU (slower)")

## 2. Configuration

In [ ]:
# === CONFIGURATION ===

# Data settings
DATA_FILE = "nn_training_data.csv"
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Network architecture
# Single TF (25 features): [50, 30, 15]
# Multi TF (75 features): [80, 50, 30]
ARCHITECTURE = [50, 30, 15]  # Adjust based on your data

# Training settings
EPOCHS = 1000
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 50
REDUCE_LR_PATIENCE = 20

# Regularization
DROPOUT_RATE = 0.3
L2_REG = 0.0001

# Output
MODEL_OUTPUT = "trained_model.h5"

print("Configuration loaded successfully!")

## 3. Load and Explore Data

In [ ]:
# Load data
print(f"Loading data from {DATA_FILE}...")
df = pd.read_csv(DATA_FILE)

print(f"✅ Loaded {len(df)} samples")
print(f"📊 Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Separate features and labels
X = df.iloc[:, :-1].values  # All columns except last
y = df.iloc[:, -1].values    # Last column (Label)

input_dim = X.shape[1]

print(f"Feature shape: {X.shape}")
print(f"Input dimension: {input_dim} features")
print(f"\nLabel distribution:")
print(f"  BUY (1.0):  {np.sum(y == 1.0)} samples ({np.sum(y == 1.0)/len(y)*100:.1f}%)")
print(f"  SELL (0.0): {np.sum(y == 0.0)} samples ({np.sum(y == 0.0)/len(y)*100:.1f}%)")

# Detect mode
if input_dim == 25:
    print("\n📊 Mode: Single Timeframe (25 features)")
    ARCHITECTURE = [50, 30, 15]
elif input_dim == 75:
    print("\n📊 Mode: Multi-Timeframe (75 features)")
    ARCHITECTURE = [80, 50, 30]
else:
    print(f"\n⚠️ Custom input dimension: {input_dim}")
    ARCHITECTURE = [input_dim*2, input_dim, input_dim//2]

print(f"Architecture: {input_dim} -> {ARCHITECTURE} -> 1")

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
labels, counts = np.unique(y, return_counts=True)
axes[0].bar(['SELL', 'BUY'], counts, color=['red', 'green'])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(counts, labels=['SELL', 'BUY'], autopct='%1.1f%%', 
            colors=['red', 'green'], startangle=90)
axes[1].set_title('Label Distribution (%)')

plt.tight_layout()
plt.show()

print(f"✅ Data balanced: {'Yes' if abs(counts[0] - counts[1]) / len(y) < 0.1 else 'No (may need class weights)'}")

## 4. Data Splitting

In [ ]:
# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Further split training into train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

print("Dataset splits:")
print(f"  Training:   {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test:       {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")
print(f"\n✅ Ready for training!")

## 5. Build Model

In [ ]:
def create_model(input_dim, architecture, dropout_rate=0.3, l2_reg=0.0001):
    """Create neural network model"""
    model = keras.Sequential(name="NeuralPredictor")
    
    # Input layer
    model.add(layers.Input(shape=(input_dim,), name="input_layer"))
    
    # Hidden layers
    for i, units in enumerate(architecture):
        if units > 0:
            model.add(layers.Dense(
                units,
                activation='swish',
                kernel_regularizer=keras.regularizers.l2(l2_reg),
                name=f"hidden_layer_{i+1}"
            ))
            model.add(layers.BatchNormalization(name=f"batch_norm_{i+1}"))
            model.add(layers.Dropout(dropout_rate, name=f"dropout_{i+1}"))
    
    # Output layer
    model.add(layers.Dense(1, activation='sigmoid', name="output_layer"))
    
    # Compile
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
    )
    
    return model

# Create model
model = create_model(input_dim, ARCHITECTURE, DROPOUT_RATE, L2_REG)
model.summary()

## 6. Train Model

In [ ]:
# Setup callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=REDUCE_LR_PATIENCE,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_checkpoint.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Train
print("Starting training...\n")
start_time = datetime.now()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

training_time = (datetime.now() - start_time).total_seconds()
print(f"\n✅ Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

## 7. Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Training History', fontsize=16, fontweight='bold')

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Train Acc', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Val Acc', linewidth=2)
axes[0, 1].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history['precision'], label='Train Precision', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Val Precision', linewidth=2)
axes[1, 0].set_title('Model Precision', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Val Recall', linewidth=2)
axes[1, 1].set_title('Model Recall', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training history plotted")

## 8. Evaluate Model

In [ ]:
# Make predictions
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_test_int = y_test.astype(int)

# Calculate metrics
accuracy = accuracy_score(y_test_int, y_pred)
precision = precision_score(y_test_int, y_pred, zero_division=0)
recall = recall_score(y_test_int, y_pred, zero_division=0)

print("="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(f"\nAccuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print("\n" + "="*50)

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test_int, y_pred, target_names=['SELL', 'BUY']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_int, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['SELL', 'BUY'], yticklabels=['SELL', 'BUY'],
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nTrue Negatives (SELL):  {cm[0,0]}")
print(f"False Positives:        {cm[0,1]}")
print(f"False Negatives:        {cm[1,0]}")
print(f"True Positives (BUY):   {cm[1,1]}")

## 9. Confidence Analysis

In [ ]:
# Analyze accuracy at different confidence thresholds
print("Confidence Threshold Analysis:")
print("="*60)

thresholds = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
results = []

for threshold in thresholds:
    confident_mask = (y_pred_prob < (1-threshold)) | (y_pred_prob > threshold)
    confident_mask = confident_mask.flatten()
    
    if np.sum(confident_mask) > 0:
        conf_y_test = y_test_int[confident_mask]
        conf_y_pred = y_pred[confident_mask]
        conf_acc = accuracy_score(conf_y_test, conf_y_pred)
        conf_count = np.sum(confident_mask)
        conf_pct = conf_count / len(y_test) * 100
        
        results.append({
            'threshold': threshold,
            'accuracy': conf_acc,
            'count': conf_count,
            'percentage': conf_pct
        })
        
        print(f"Confidence >{threshold:.2f}: {conf_acc*100:5.2f}% accuracy | "
              f"{conf_count:4d} signals ({conf_pct:5.1f}%)")

# Plot confidence analysis
if results:
    df_results = pd.DataFrame(results)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy vs threshold
    axes[0].plot(df_results['threshold'], df_results['accuracy']*100, 
                marker='o', linewidth=2, markersize=8)
    axes[0].set_xlabel('Confidence Threshold', fontsize=12)
    axes[0].set_ylabel('Accuracy (%)', fontsize=12)
    axes[0].set_title('Accuracy vs Confidence Threshold', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([50, 100])
    
    # Signal percentage vs threshold
    axes[1].plot(df_results['threshold'], df_results['percentage'], 
                marker='s', linewidth=2, markersize=8, color='orange')
    axes[1].set_xlabel('Confidence Threshold', fontsize=12)
    axes[1].set_ylabel('% of Signals', fontsize=12)
    axes[1].set_title('Signal Coverage vs Confidence Threshold', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('confidence_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Confidence analysis complete")

## 10. Prediction Distribution

In [ ]:
# Plot prediction probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All predictions
axes[0].hist(y_pred_prob, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Decision Boundary')
axes[0].set_xlabel('Predicted Probability (BUY)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Predictions', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Separate by true label
buy_mask = y_test == 1.0
sell_mask = y_test == 0.0

axes[1].hist(y_pred_prob[buy_mask], bins=30, alpha=0.6, label='True BUY', color='green', edgecolor='black')
axes[1].hist(y_pred_prob[sell_mask], bins=30, alpha=0.6, label='True SELL', color='red', edgecolor='black')
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Boundary')
axes[1].set_xlabel('Predicted Probability (BUY)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Predictions by True Label', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Prediction distribution plotted")

## 11. Save Model

In [ ]:
# Save trained model
model.save(MODEL_OUTPUT)
print(f"✅ Model saved to: {MODEL_OUTPUT}")
print(f"\nModel size: {os.path.getsize(MODEL_OUTPUT) / 1024 / 1024:.2f} MB")
print(f"\nNext steps:")
print(f"1. Run weight converter: python convert_weights.py")
print(f"2. Copy converted weights to MT5 EA")
print(f"3. Test in Strategy Tester!")

## 12. Experiment (Optional)

Try different configurations and compare results:

In [ ]:
# Example: Try different learning rates
learning_rates = [0.01, 0.001, 0.0001]
results_comparison = []

for lr in learning_rates:
    print(f"\nTesting learning rate: {lr}")
    
    # Create and train model
    test_model = create_model(input_dim, ARCHITECTURE, DROPOUT_RATE, L2_REG)
    test_model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    history = test_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,  # Shorter for comparison
        batch_size=BATCH_SIZE,
        verbose=0
    )
    
    # Evaluate
    y_pred = (test_model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test.astype(int), y_pred)
    
    results_comparison.append({'lr': lr, 'accuracy': acc})
    print(f"Accuracy: {acc*100:.2f}%")

# Plot comparison
df_comp = pd.DataFrame(results_comparison)
plt.figure(figsize=(8, 5))
plt.bar(range(len(df_comp)), df_comp['accuracy']*100)
plt.xticks(range(len(df_comp)), df_comp['lr'])
plt.xlabel('Learning Rate')
plt.ylabel('Accuracy (%)')
plt.title('Learning Rate Comparison')
plt.grid(axis='y', alpha=0.3)
plt.show()